# Regression with Amazon SageMaker XGBoost algorithm (V3 - sagemaker-core)


---

This notebook's CI test result for us-west-2 is as follows. CI test results in other regions can be found at the end of the notebook. 

![This us-west-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-west-2/build_and_train_models|sm-regression_xgboost|sm-regression_xgboost.ipynb)

---

> **SageMaker Python SDK v3 note:** This notebook has been migrated to the SageMaker Python SDK **v3** using the `sagemaker-core` resource-class APIs (`TrainingJob`, `HyperParameterTuningJob`, `Model`, `EndpointConfig`, `Endpoint`). It no longer uses raw `boto3` low-level `client.create_*` calls. Install with `pip install sagemaker` (v3) or `pip install sagemaker-core`.

---

_**Single machine training for regression with Amazon SageMaker XGBoost algorithm**_

---

---
## Contents
1. [Introduction](#Introduction)
2. [Setup](#Setup)
  1. [Fetching the dataset](#Fetching-the-dataset)
  2. [Data Ingestion](#Data-ingestion)
3. [Training the XGBoost model](#Training-the-XGBoost-model)
  1. [Plotting evaluation metrics](#Plotting-evaluation-metrics)
4. [Set up hosting for the model](#Set-up-hosting-for-the-model)
  1. [Import model into hosting](#Import-model-into-hosting)
  2. [Create endpoint configuration](#Create-endpoint-configuration)
  3. [Create endpoint](#Create-endpoint)
5. [Validate the model for use](#Validate-the-model-for-use)

---
## Introduction

This notebook demonstrates the use of Amazon SageMaker’s implementation of the XGBoost algorithm to train and host a regression model. We use the [Abalone data](https://www.csie.ntu.edu.tw/~cjlin/libsvmtools/datasets/regression.html) originally from the UCI data repository [1]. More details about the original dataset can be found [here](https://archive.ics.uci.edu/ml/machine-learning-databases/abalone/abalone.names).  In the libsvm converted [version](https://www.csie.ntu.edu.tw/~cjlin/libsvmtools/datasets/regression.html), the nominal feature (Male/Female/Infant) has been converted into a real valued feature. Age of abalone is to be predicted from eight physical measurements. Dataset is already processed and stored on S3. Scripts used for processing the data can be found in the [Appendix](#Appendix). These include downloading the data, splitting into train, validation and test, and uploading to S3 bucket. 

>[1] Dua, D. and Graff, C. (2019). UCI Machine Learning Repository [http://archive.ics.uci.edu/ml]. Irvine, CA: University of California, School of Information and Computer Science.

## Setup


This notebook was tested in Amazon SageMaker Studio on a ml.t3.medium instance with Python 3 (Data Science) kernel.

Let's start by specifying:
1. The S3 buckets and prefixes that you want to use for saving the model and where training data is located. This should be within the same region as the Notebook Instance, training, and hosting. 
1. The IAM role arn used to give training and hosting access to your data. See the documentation for how to create these. Note, if more than one role is required for notebook instances, training, and/or hosting, please replace the boto regexp with a the appropriate full IAM role arn string(s).

In [1]:
# [papermill-run] pip install disabled to use local staging V3 SDK

In [2]:
%%time

import os
import boto3
import re
from sagemaker.core.helper.session_helper import Session, get_execution_role

import boto3 as _boto3
sagemaker_session = Session(boto_session=_boto3.Session(region_name="us-west-1"))
role = "arn:aws:iam::729646638167:role/SageMakerRole"  # [papermill-run] explicit role
region = "us-west-1"  # [papermill-run] explicit region
s3_client = boto3.client("s3")


# S3 bucket where the training data is located.
data_bucket = f"sagemaker-sample-files"
data_prefix = "datasets/tabular/uci_abalone"
data_bucket_path = f"s3://{data_bucket}"

# S3 bucket for saving code and model artifacts.
# Feel free to specify a different bucket and prefix
output_bucket = sagemaker_session.default_bucket()
output_prefix = "sagemaker/DEMO-xgboost-abalone-default"
output_bucket_path = f"s3://{output_bucket}"
default_bucket_prefix = sagemaker_session.default_bucket_prefix

# If a default bucket prefix is specified, append it to the s3 path
if default_bucket_prefix:
    output_prefix = f"{default_bucket_prefix}/{output_prefix}"

for data_category in ["train", "test", "validation"]:
    data_key = "{0}/{1}/abalone.{1}".format(data_prefix, data_category)
    output_key = "{0}/{1}/abalone.{1}".format(output_prefix, data_category)
    data_filename = "abalone.{}".format(data_category)
    s3_client.download_file(data_bucket, data_key, data_filename)
    s3_client.upload_file(data_filename, output_bucket, output_key)

[07/09/26 12:02:13] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=8556643;file:///Users/lucasjia/Library/Python/3.9/lib/python/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=8556644;file:///Users/lucasjia/Library/Python/3.9/lib/python/site-packages/botocore/credentials.py#1392\1392]8;;\

sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /Users/lucasjia/Library/Application Support/sagemaker/config.yaml


/Users/lucasjia/Library/Python/3.9/lib/python/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


[07/09/26 12:02:14] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=8556649;file:///Users/lucasjia/Library/Python/3.9/lib/python/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=8556650;file:///Users/lucasjia/Library/Python/3.9/lib/python/site-packages/botocore/credentials.py#1392\1392]8;;\

CPU times: user 2.41 s, sys: 2.82 s, total: 5.23 s
Wall time: 7 s


## Training the XGBoost model

After setting training parameters, we kick off training, and poll for status until training is completed, which in this example, takes between 5 and 6 minutes.

Training can be done by either calling SageMaker Training with a set of hyperparameters values to train with, or by leveraging SageMaker Automatic Model Tuning ([AMT](https://docs.aws.amazon.com/sagemaker/latest/dg/automatic-model-tuning.html)). AMT, also known as hyperparameter tuning (HPO), finds the best version of a model by running many training jobs on your dataset using the algorithm and ranges of hyperparameters that you specify. It then chooses the hyperparameter values that result in a model that performs the best, as measured by a metric that you choose.

In this notebook, both methods are used for demonstration purposes, but the model that the HPO job creates is the one that is eventually hosted. You can instead choose to deploy the model created by the standalone training job by changing the below variable `deploy_amt_model` to False.

### Initiliazing common variables 

In [3]:
from sagemaker.core import image_uris

container = image_uris.retrieve("xgboost", region, "1.7-1")
deploy_amt_model = True

[07/09/26 12:02:17] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=8556655;file:///Users/lucasjia/Library/Python/3.9/lib/python/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=8556656;file:///Users/lucasjia/Library/Python/3.9/lib/python/site-packages/botocore/credentials.py#1392\1392]8;;\

[07/09/26 12:02:18] INFO     Ignoring unnecessary instance type: None.                            ]8;id=8556663;file:///Users/lucasjia/pysdk/sagemaker-python-sdk-staging-lucas/sagemaker-core/src/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=8556664;file:///Users/lucasjia/pysdk/sagemaker-python-sdk-staging-lucas/sagemaker-core/src/sagemaker/core/image_uris.py#535\535]8;;\

### Training with SageMaker Training

In [4]:
%%time
from time import gmtime, strftime
from sagemaker.core.resources import TrainingJob
from sagemaker.core.shapes import (
    AlgorithmSpecification,
    Channel,
    DataSource,
    S3DataSource,
    ResourceConfig,
    StoppingCondition,
    OutputDataConfig,
)

training_job_name = f"DEMO-xgboost-regression-{strftime('%Y-%m-%d-%H-%M-%S', gmtime())}"

print(
    f"Creating a training job with name: {training_job_name}. "
    "It will take between 5 and 6 minutes to complete."
)

training_job = TrainingJob.create(
    training_job_name=training_job_name,
    hyper_parameters={
        "max_depth": "5",
        "eta": "0.2",
        "gamma": "4",
        "min_child_weight": "6",
        "subsample": "0.7",
        "objective": "reg:linear",
        "num_round": "50",
        "verbosity": "2",
    },
    algorithm_specification=AlgorithmSpecification(
        training_image=container,
        training_input_mode="File",
    ),
    role_arn=role,
    input_data_config=[
        Channel(
            channel_name="train",
            content_type="libsvm",
            compression_type="None",
            data_source=DataSource(
                s3_data_source=S3DataSource(
                    s3_data_type="S3Prefix",
                    s3_uri=f"{output_bucket_path}/{output_prefix}/train",
                    s3_data_distribution_type="FullyReplicated",
                )
            ),
        ),
        Channel(
            channel_name="validation",
            content_type="libsvm",
            compression_type="None",
            data_source=DataSource(
                s3_data_source=S3DataSource(
                    s3_data_type="S3Prefix",
                    s3_uri=f"{output_bucket_path}/{output_prefix}/validation",
                    s3_data_distribution_type="FullyReplicated",
                )
            ),
        ),
    ],
    output_data_config=OutputDataConfig(
        s3_output_path=f"{output_bucket_path}/{output_prefix}/single-xgboost"
    ),
    resource_config=ResourceConfig(
        instance_type="ml.m5.2xlarge",
        instance_count=1,
        volume_size_in_gb=5,
    ),
    stopping_condition=StoppingCondition(max_runtime_in_seconds=3600),
)

# Wait for the training job to complete.
training_job.wait(logs=False)
print(training_job.training_job_status)

Creating a training job with name: DEMO-xgboost-regression-2026-07-09-19-02-18. It will take between 5 and 6 minutes to complete.
sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /Users/lucasjia/Library/Application Support/sagemaker/config.yaml


                    INFO     Creating training_job resource.                                     ]8;id=8556671;file:///Users/lucasjia/pysdk/sagemaker-python-sdk-staging-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=8556672;file:///Users/lucasjia/pysdk/sagemaker-python-sdk-staging-lucas/sagemaker-core/src/sagemaker/core/resources.py#31116\31116]8;;\

                    WARNING  No region provided. Using default region.                                 ]8;id=8556679;file:///Users/lucasjia/pysdk/sagemaker-python-sdk-staging-lucas/sagemaker-core/src/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=8556680;file:///Users/lucasjia/pysdk/sagemaker-python-sdk-staging-lucas/sagemaker-core/src/sagemaker/core/utils/utils.py#361\361]8;;\

                    INFO     Runs on sagemaker prod, region:us-west-1                                  ]8;id=8556686;file:///Users/lucasjia/pysdk/sagemaker-python-sdk-staging-lucas/sagemaker-core/src/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=8556687;file:///Users/lucasjia/pysdk/sagemaker-python-sdk-staging-lucas/sagemaker-core/src/sagemaker/core/utils/utils.py#375\375]8;;\

                    INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=8556692;file:///Users/lucasjia/Library/Python/3.9/lib/python/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=8556693;file:///Users/lucasjia/Library/Python/3.9/lib/python/site-packages/botocore/credentials.py#1392\1392]8;;\

                    INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=8556698;file:///Users/lucasjia/Library/Python/3.9/lib/python/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=8556699;file:///Users/lucasjia/Library/Python/3.9/lib/python/site-packages/botocore/credentials.py#1392\1392]8;;\

Output()

[07/09/26 12:04:41] INFO     Final Resource Status: Completed                                    ]8;id=8556705;file:///Users/lucasjia/pysdk/sagemaker-python-sdk-staging-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=8556706;file:///Users/lucasjia/pysdk/sagemaker-python-sdk-staging-lucas/sagemaker-core/src/sagemaker/core/resources.py#31442\31442]8;;\

Completed
CPU times: user 1.87 s, sys: 342 ms, total: 2.21 s
Wall time: 2min 23s


Note that the "validation" channel has been initialized too. The SageMaker XGBoost algorithm actually calculates RMSE and writes it to the CloudWatch logs on the data passed to the "validation" channel.

### Tuning with SageMaker Automatic Model Tuning

To create a tuning job using the AWS SageMaker Automatic Model Tuning API, you need to define 3 attributes. 

1. the tuning job name (string)
2. the tuning job config (to specify settings for the hyperparameter tuning job - JSON object)
3. training job definition (to configure the training jobs that the tuning job launches - JSON object).

To learn more about that, refer to the [Configure and Launch a Hyperparameter Tuning Job](https://docs.aws.amazon.com/sagemaker/latest/dg/automatic-model-tuning-ex-tuning-job.html) documentation.

Note that the tuning job will 12-17 minutes to complete.

In [5]:
from time import gmtime, strftime
from sagemaker.core.resources import HyperParameterTuningJob
from sagemaker.core.shapes import (
    HyperParameterTuningJobConfig,
    ResourceLimits,
    ParameterRanges,
    ContinuousParameterRange,
    IntegerParameterRange,
    HyperParameterTrainingJobDefinition,
    HyperParameterTuningJobObjective,
    HyperParameterAlgorithmSpecification,
)

tuning_job_name = "DEMO-xgboost-reg-" + strftime("%d-%H-%M-%S", gmtime())

tuning_job_config = HyperParameterTuningJobConfig(
    strategy="Bayesian",
    hyper_parameter_tuning_job_objective=HyperParameterTuningJobObjective(
        metric_name="validation:rmse",
        type="Minimize",
    ),
    resource_limits=ResourceLimits(
        # Increase the max number of training jobs for increased accuracy (and training time).
        max_number_of_training_jobs=6,
        # Change parallel training jobs run by AMT to reduce total training time.
        # Constrained by your account limits.
        # if max_jobs == max_parallel_jobs then Bayesian search turns to Random.
        max_parallel_training_jobs=2,
    ),
    parameter_ranges=ParameterRanges(
        continuous_parameter_ranges=[
            ContinuousParameterRange(name="eta", min_value="0.1", max_value="0.5"),
            ContinuousParameterRange(name="gamma", min_value="0", max_value="5"),
            ContinuousParameterRange(name="min_child_weight", min_value="0", max_value="120"),
            ContinuousParameterRange(name="subsample", min_value="0.5", max_value="1"),
            ContinuousParameterRange(name="alpha", min_value="0", max_value="2"),
        ],
        integer_parameter_ranges=[
            IntegerParameterRange(name="max_depth", min_value="0", max_value="10"),
            IntegerParameterRange(name="num_round", min_value="1", max_value="4000"),
        ],
    ),
)

training_job_definition = HyperParameterTrainingJobDefinition(
    algorithm_specification=HyperParameterAlgorithmSpecification(
        training_image=container,
        training_input_mode="File",
    ),
    input_data_config=[
        Channel(
            channel_name="train",
            content_type="libsvm",
            compression_type="None",
            data_source=DataSource(
                s3_data_source=S3DataSource(
                    s3_data_type="S3Prefix",
                    s3_uri=f"{output_bucket_path}/{output_prefix}/train",
                    s3_data_distribution_type="FullyReplicated",
                )
            ),
        ),
        Channel(
            channel_name="validation",
            content_type="libsvm",
            compression_type="None",
            data_source=DataSource(
                s3_data_source=S3DataSource(
                    s3_data_type="S3Prefix",
                    s3_uri=f"{output_bucket_path}/{output_prefix}/validation",
                    s3_data_distribution_type="FullyReplicated",
                )
            ),
        ),
    ],
    output_data_config=OutputDataConfig(
        s3_output_path=f"{output_bucket_path}/{output_prefix}/single-xgboost"
    ),
    resource_config=ResourceConfig(
        instance_type="ml.m5.2xlarge",
        instance_count=1,
        volume_size_in_gb=5,
    ),
    role_arn=role,
    static_hyper_parameters={
        "objective": "reg:linear",
        "verbosity": "2",
    },
    stopping_condition=StoppingCondition(max_runtime_in_seconds=43200),
)

print(
    f"Creating a tuning job with name: {tuning_job_name}. "
    "It will take between 12 and 17 minutes to complete."
)

tuning_job = HyperParameterTuningJob.create(
    hyper_parameter_tuning_job_name=tuning_job_name,
    hyper_parameter_tuning_job_config=tuning_job_config,
    training_job_definition=training_job_definition,
)

tuning_job.wait()
print(tuning_job.hyper_parameter_tuning_job_status)

Creating a tuning job with name: DEMO-xgboost-reg-09-19-04-43. It will take between 12 and 17 minutes to complete.


                    INFO     Creating hyper_parameter_tuning_job resource.                       ]8;id=8560050;file:///Users/lucasjia/pysdk/sagemaker-python-sdk-staging-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=8560051;file:///Users/lucasjia/pysdk/sagemaker-python-sdk-staging-lucas/sagemaker-core/src/sagemaker/core/resources.py#14857\14857]8;;\

Output()

[07/09/26 12:09:19] INFO     Final Resource Status: Completed                                    ]8;id=8560057;file:///Users/lucasjia/pysdk/sagemaker-python-sdk-staging-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=8560058;file:///Users/lucasjia/pysdk/sagemaker-python-sdk-staging-lucas/sagemaker-core/src/sagemaker/core/resources.py#15086\15086]8;;\

Completed


## Set up hosting for the model
In order to set up hosting, we have to import the model from training to hosting. 

### Import model into hosting

Register the model with hosting. This allows the flexibility of importing models trained elsewhere.

In [6]:
%%time
from sagemaker.core.resources import Model
from sagemaker.core.shapes import ContainerDefinition

if deploy_amt_model:
    training_of_model_to_be_hosted = tuning_job.best_training_job.training_job_name
else:
    training_of_model_to_be_hosted = training_job_name

model_name = f"{training_of_model_to_be_hosted}-model"
print(model_name)

# Fetch the S3 location of the trained model artifacts from the training job.
best_training_job = TrainingJob.get(training_of_model_to_be_hosted)
model_data = best_training_job.model_artifacts.s3_model_artifacts
print(model_data)

xgboost_model = Model.create(
    model_name=model_name,
    primary_container=ContainerDefinition(
        image=container,
        model_data_url=model_data,
    ),
    execution_role_arn=role,
)
print(xgboost_model.get_name())

DEMO-xgboost-reg-09-19-04-43-006-d0bdfa00-model
s3://sagemaker-us-west-1-729646638167/sagemaker/DEMO-xgboost-abalone-default/single-xgboost/DEMO-xgboost-reg-09-19-04-43-006-d0bdfa00/output/model.tar.gz


                    INFO     Creating model resource.                                            ]8;id=8566508;file:///Users/lucasjia/pysdk/sagemaker-python-sdk-staging-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=8566509;file:///Users/lucasjia/pysdk/sagemaker-python-sdk-staging-lucas/sagemaker-core/src/sagemaker/core/resources.py#20477\20477]8;;\

DEMO-xgboost-reg-09-19-04-43-006-d0bdfa00-model
CPU times: user 22.9 ms, sys: 5.98 ms, total: 28.8 ms
Wall time: 698 ms


### Create endpoint configuration

SageMaker supports configuring REST endpoints in hosting with multiple models, e.g. for A/B testing purposes. In order to support this, customers create an endpoint configuration, that describes the distribution of traffic across the models, whether split, shadowed, or sampled in some way. In addition, the endpoint configuration describes the instance type required for model deployment.

In [7]:
from time import gmtime, strftime
from sagemaker.core.resources import EndpointConfig
from sagemaker.core.shapes import ProductionVariant

endpoint_config_name = f"DEMO-XGBoostEndpointConfig-{strftime('%Y-%m-%d-%H-%M-%S', gmtime())}"
print(f"Creating endpoint config with name: {endpoint_config_name}.")

endpoint_config = EndpointConfig.create(
    endpoint_config_name=endpoint_config_name,
    production_variants=[
        ProductionVariant(
            variant_name="AllTraffic",
            model_name=model_name,
            instance_type="ml.m5.xlarge",
            initial_instance_count=1,
            initial_variant_weight=1,
        )
    ],
)

print(f"Endpoint Config Arn: {endpoint_config.endpoint_config_arn}")

Creating endpoint config with name: DEMO-XGBoostEndpointConfig-2026-07-09-19-09-21.


[07/09/26 12:09:21] INFO     Creating endpoint_config resource.                                  ]8;id=8566515;file:///Users/lucasjia/pysdk/sagemaker-python-sdk-staging-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=8566516;file:///Users/lucasjia/pysdk/sagemaker-python-sdk-staging-lucas/sagemaker-core/src/sagemaker/core/resources.py#11069\11069]8;;\

Endpoint Config Arn: arn:aws:sagemaker:us-west-1:729646638167:endpoint-config/DEMO-XGBoostEndpointConfig-2026-07-09-19-09-21


### Create endpoint
Lastly, the customer creates the endpoint that serves up the model, through specifying the name and configuration defined above. The end result is an endpoint that can be validated and incorporated into production applications. This takes 9-11 minutes to complete.

In [8]:
%%time
from sagemaker.core.resources import Endpoint

endpoint_name = f'DEMO-XGBoostEndpoint-{strftime("%Y-%m-%d-%H-%M-%S", gmtime())}'
print(
    f"Creating endpoint with name: {endpoint_name}. "
    "This will take between 9 and 11 minutes to complete."
)

xgboost_endpoint = Endpoint.create(
    endpoint_name=endpoint_name,
    endpoint_config_name=endpoint_config_name,
)

# Wait until the endpoint is InService before invoking it.
xgboost_endpoint.wait_for_status("InService")
print(f"Arn: {xgboost_endpoint.endpoint_arn}")
print(f"Status: {xgboost_endpoint.endpoint_status}")

Creating endpoint with name: DEMO-XGBoostEndpoint-2026-07-09-19-09-21. This will take between 9 and 11 minutes to complete.


                    INFO     Creating endpoint resource.                                         ]8;id=8566522;file:///Users/lucasjia/pysdk/sagemaker-python-sdk-staging-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=8566523;file:///Users/lucasjia/pysdk/sagemaker-python-sdk-staging-lucas/sagemaker-core/src/sagemaker/core/resources.py#10228\10228]8;;\

Output()

[07/09/26 12:12:21] INFO     Final Resource Status: InService                                    ]8;id=8566529;file:///Users/lucasjia/pysdk/sagemaker-python-sdk-staging-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=8566530;file:///Users/lucasjia/pysdk/sagemaker-python-sdk-staging-lucas/sagemaker-core/src/sagemaker/core/resources.py#10484\10484]8;;\

Arn: arn:aws:sagemaker:us-west-1:729646638167:endpoint/DEMO-XGBoostEndpoint-2026-07-09-19-09-21
Status: InService
CPU times: user 2.09 s, sys: 272 ms, total: 2.37 s
Wall time: 2min 59s


## Validate the model for use
Finally, the customer can now validate the model for use. We invoke the endpoint directly through the
`Endpoint` resource object created above (`endpoint.invoke(...)`), and generate predictions from the
trained model using that endpoint.

In [9]:
from sagemaker.core.resources import Endpoint

# Re-hydrate the Endpoint resource object (useful if resuming the notebook).
xgboost_endpoint = Endpoint.get(endpoint_name)

Download test data

In [10]:
FILE_TEST = "abalone.test"
s3 = boto3.client("s3")
s3.download_file(data_bucket, f"{data_prefix}/test/{FILE_TEST}", FILE_TEST)

Start with a single prediction.

In [11]:
!head -1 abalone.test > abalone.single.test

In [12]:
%%time
import math

file_name = "abalone.single.test"  # customize to your test file
with open(file_name, "r") as f:
    payload = f.read().strip()

response = xgboost_endpoint.invoke(body=payload, content_type="text/x-libsvm")
result = response.body.read().decode("utf-8")
result = result.split(",")
result = [math.ceil(float(i)) for i in result]
label = payload.strip(" ").split()[0]
print(f"Label: {label}\nPrediction: {result[0]}")

Label: 12
Prediction: 13
CPU times: user 7.15 ms, sys: 2.83 ms, total: 9.98 ms
Wall time: 316 ms


OK, a single prediction works. Let's do a whole batch to see how good is the predictions accuracy.

In [13]:
import sys
import math


def do_predict(data, endpoint, content_type):
    payload = "\n".join(data)
    response = endpoint.invoke(body=payload, content_type=content_type)
    result = response.body.read()
    result = result.decode("utf-8")
    result = result.strip("\n").split("\n")
    preds = [float(num) for num in result]
    preds = [math.ceil(num) for num in preds]
    return preds


def batch_predict(data, batch_size, endpoint, content_type):
    items = len(data)
    arrs = []

    for offset in range(0, items, batch_size):
        if offset + batch_size < items:
            results = do_predict(data[offset : (offset + batch_size)], endpoint, content_type)
            arrs.extend(results)
        else:
            arrs.extend(do_predict(data[offset:items], endpoint, content_type))
        sys.stdout.write(".")
    return arrs

The following helps us calculate the Median Absolute Percent Error (MdAPE) on the batch dataset. 

In [14]:
%%time
import numpy as np

with open(FILE_TEST, "r") as f:
    payload = f.read().strip()

labels = [int(line.split(" ")[0]) for line in payload.split("\n")]
test_data = [line for line in payload.split("\n")]
preds = batch_predict(test_data, 100, xgboost_endpoint, "text/x-libsvm")

print(
    "\n Median Absolute Percent Error (MdAPE) = ",
    np.median(np.abs(np.array(labels) - np.array(preds)) / np.array(labels)),
)

..

..

..

.
 Median Absolute Percent Error (MdAPE) =  0.1
CPU times: user 19.6 ms, sys: 2.06 ms, total: 21.7 ms
Wall time: 467 ms


### Delete Endpoint
Once you are done using the endpoint, you can use the following to delete it. 

In [15]:
xgboost_endpoint.delete()

[07/09/26 12:12:25] INFO     Deleting Endpoint - DEMO-XGBoostEndpoint-2026-07-09-19-09-21        ]8;id=8570736;file:///Users/lucasjia/pysdk/sagemaker-python-sdk-staging-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=8570737;file:///Users/lucasjia/pysdk/sagemaker-python-sdk-staging-lucas/sagemaker-core/src/sagemaker/core/resources.py#10428\10428]8;;\

## Appendix

### Data split and upload

Following methods split the data into train/test/validation datasets and upload files to S3.

In [16]:
import io
import boto3
import random


def data_split(
    FILE_DATA,
    FILE_TRAIN,
    FILE_VALIDATION,
    FILE_TEST,
    PERCENT_TRAIN,
    PERCENT_VALIDATION,
    PERCENT_TEST,
):
    data = [l for l in open(FILE_DATA, "r")]
    train_file = open(FILE_TRAIN, "w")
    valid_file = open(FILE_VALIDATION, "w")
    tests_file = open(FILE_TEST, "w")

    num_of_data = len(data)
    num_train = int((PERCENT_TRAIN / 100.0) * num_of_data)
    num_valid = int((PERCENT_VALIDATION / 100.0) * num_of_data)
    num_tests = int((PERCENT_TEST / 100.0) * num_of_data)

    data_fractions = [num_train, num_valid, num_tests]
    split_data = [[], [], []]

    rand_data_ind = 0

    for split_ind, fraction in enumerate(data_fractions):
        for i in range(fraction):
            rand_data_ind = random.randint(0, len(data) - 1)
            split_data[split_ind].append(data[rand_data_ind])
            data.pop(rand_data_ind)

    for l in split_data[0]:
        train_file.write(l)

    for l in split_data[1]:
        valid_file.write(l)

    for l in split_data[2]:
        tests_file.write(l)

    train_file.close()
    valid_file.close()
    tests_file.close()


def write_to_s3(fobj, bucket, key):
    return (
        boto3.Session(region_name=region)
        .resource("s3")
        .Bucket(bucket)
        .Object(key)
        .upload_fileobj(fobj)
    )


def upload_to_s3(bucket, channel, filename):
    fobj = open(filename, "rb")
    key = f"{prefix}/{channel}"
    url = f"s3://{bucket}/{key}/{filename}"
    print(f"Writing to {url}")
    write_to_s3(fobj, bucket, key)

### Data ingestion

Next, we read the dataset from the existing repository into memory, for preprocessing prior to training. This processing could be done *in situ* by Amazon Athena, Apache Spark in Amazon EMR, Amazon Redshift, etc., assuming the dataset is present in the appropriate location. Then, the next step would be to transfer the data to S3 for use in training. For small datasets, such as this one, reading into memory isn't onerous, though it would be for larger datasets.

In [17]:
%%time
s3 = boto3.client("s3")

bucket = sagemaker_session.default_bucket()
prefix = "sagemaker/DEMO-xgboost-abalone-default"
# Load the dataset
FILE_DATA = "abalone"
s3.download_file(
    f"sagemaker-example-files-prod-{region}",
    f"datasets/tabular/uci_abalone/abalone.libsvm",
    FILE_DATA,
)

# split the downloaded data into train/test/validation files
FILE_TRAIN = "abalone.train"
FILE_VALIDATION = "abalone.validation"
FILE_TEST = "abalone.test"
PERCENT_TRAIN = 70
PERCENT_VALIDATION = 15
PERCENT_TEST = 15
data_split(
    FILE_DATA,
    FILE_TRAIN,
    FILE_VALIDATION,
    FILE_TEST,
    PERCENT_TRAIN,
    PERCENT_VALIDATION,
    PERCENT_TEST,
)

# upload the files to the S3 bucket
upload_to_s3(bucket, "train", FILE_TRAIN)
upload_to_s3(bucket, "validation", FILE_VALIDATION)
upload_to_s3(bucket, "test", FILE_TEST)

Writing to s3://sagemaker-us-west-1-729646638167/sagemaker/DEMO-xgboost-abalone-default/train/abalone.train


                    INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=8570742;file:///Users/lucasjia/Library/Python/3.9/lib/python/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=8570743;file:///Users/lucasjia/Library/Python/3.9/lib/python/site-packages/botocore/credentials.py#1392\1392]8;;\

Writing to s3://sagemaker-us-west-1-729646638167/sagemaker/DEMO-xgboost-abalone-default/validation/abalone.validation


[07/09/26 12:12:26] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=8570748;file:///Users/lucasjia/Library/Python/3.9/lib/python/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=8570749;file:///Users/lucasjia/Library/Python/3.9/lib/python/site-packages/botocore/credentials.py#1392\1392]8;;\

Writing to s3://sagemaker-us-west-1-729646638167/sagemaker/DEMO-xgboost-abalone-default/test/abalone.test


                    INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=8570754;file:///Users/lucasjia/Library/Python/3.9/lib/python/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=8570755;file:///Users/lucasjia/Library/Python/3.9/lib/python/site-packages/botocore/credentials.py#1392\1392]8;;\

CPU times: user 165 ms, sys: 272 ms, total: 437 ms
Wall time: 1.99 s


## Notebook CI Test Results

This notebook was tested in multiple regions. The test results are as follows, except for us-west-2 which is shown at the top of the notebook.

![This us-east-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-east-1/build_and_train_models|sm-regression_xgboost|sm-regression_xgboost.ipynb)

![This us-east-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-east-2/build_and_train_models|sm-regression_xgboost|sm-regression_xgboost.ipynb)

![This us-west-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-west-1/build_and_train_models|sm-regression_xgboost|sm-regression_xgboost.ipynb)

![This ca-central-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ca-central-1/build_and_train_models|sm-regression_xgboost|sm-regression_xgboost.ipynb)

![This sa-east-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/sa-east-1/build_and_train_models|sm-regression_xgboost|sm-regression_xgboost.ipynb)

![This eu-west-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-1/build_and_train_models|sm-regression_xgboost|sm-regression_xgboost.ipynb)

![This eu-west-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-2/build_and_train_models|sm-regression_xgboost|sm-regression_xgboost.ipynb)

![This eu-west-3 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-3/build_and_train_models|sm-regression_xgboost|sm-regression_xgboost.ipynb)

![This eu-central-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-central-1/build_and_train_models|sm-regression_xgboost|sm-regression_xgboost.ipynb)

![This eu-north-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-north-1/build_and_train_models|sm-regression_xgboost|sm-regression_xgboost.ipynb)

![This ap-southeast-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-southeast-1/build_and_train_models|sm-regression_xgboost|sm-regression_xgboost.ipynb)

![This ap-southeast-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-southeast-2/build_and_train_models|sm-regression_xgboost|sm-regression_xgboost.ipynb)

![This ap-northeast-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-northeast-1/build_and_train_models|sm-regression_xgboost|sm-regression_xgboost.ipynb)

![This ap-northeast-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-northeast-2/build_and_train_models|sm-regression_xgboost|sm-regression_xgboost.ipynb)

![This ap-south-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-south-1/build_and_train_models|sm-regression_xgboost|sm-regression_xgboost.ipynb)
